Extract and consolidate fastF1 data

In [16]:
import fastf1
import pandas as pd
import os
import time
from pathlib import Path

#Project Root
PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name =="Notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
#ano de coleta
YEAR = 2018

CACHE_DIR = PROJECT_DIR / "Data" / "Cache"
OUTPUT_DIR = PROJECT_DIR / "Data" / 'Raw'

CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

fastf1.Cache.configure(cache_dir=str(CACHE_DIR))

output_file = OUTPUT_DIR / f"result_{YEAR}.parquet"

# Avoiding re-collecting a year that has already been saved
if  os.path.exists(output_file):
    print(f"Season {YEAR} already collected!")
else:
    print(f" Season {YEAR} collecting initiated!")
    try:

        schedule = fastf1.get_event_schedule(
            YEAR,
            include_testing=False
        )
    except fastf1.req.RateLimitExceededError:
        print("Request limit exceeded while fetching the calendar.")
        print("Wait before trying again.")
        schedule = None

    except Exception as e:
        print(f"Error fetching the calendar: {e}")
        schedule = None
    if schedule is not None:
        season_results = []
    
        for _, event in schedule.iterrows():
        
            round_number = int(event["RoundNumber"])
    
            if round_number == 0:
                continue
            # ignore futures events
    
            if pd.Timestamp(event["EventDate"]) > pd.Timestamp.now():
                continue
            
            print(f"Collecting GP {round_number}: {event['EventName']}")

            try:
                session = fastf1.get_session(
                    YEAR,
                    round_number,
                    "R"
                )

                session.load(
                    laps=False,
                    telemetry=False,
                    weather=False
                )

                results = session.results.copy()

                if results.empty:
                    continue


                for _, driver in results.iterrows():

                    season_results.append({
                        "Season": YEAR,
                        "Round": round_number,
                        "Date": event["EventDate"],
                        "Circuit": event["Location"],
                        "Event": event["EventName"],
                        "Driver": driver["Abbreviation"],
                        "Team": driver["TeamName"],
                        "GridPosition": driver["GridPosition"],
                        "Position": driver["Position"]
                    })
            except fastf1.req.RateLimitExceededError:
                print(" Requisition limit exceeded")
                print("Saving collected date until now")
                break

            except Exception as e:
                print(f"Error GP {round_number}: {e}")

            time.sleep(1)


        if season_results:

            df = pd.DataFrame(season_results)

            df.to_parquet(
                output_file,
                index=False
            )

            print(f"\nSeason {YEAR} saved!")
            print(f"shape: {len(df)}")

    else: 
            ("no data was collected")

print("\n Finished!")

Request for URL https://raw.githubusercontent.com/theOehrly/f1schedule/master/schedule_2018.json failed; using cached response
Traceback (most recent call last):
  File "c:\Users\vitin\VS_CODE\Canal YT\f1-podium-predictor\.venv\Lib\site-packages\requests_cache\session.py", line 311, in _resend
    response = self._send_and_cache(request, actions, cached_response, **kwargs)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\vitin\VS_CODE\Canal YT\f1-podium-predictor\.venv\Lib\site-packages\requests_cache\session.py", line 279, in _send_and_cache
    response = super().send(request, **kwargs)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\vitin\VS_CODE\Canal YT\f1-podium-predictor\.venv\Lib\site-packages\fastf1\req.py", line 129, in send
    lim.limit()
  File "c:\Users\vitin\VS_CODE\Canal YT\f1-podium-predictor\.venv\Lib\site-packages\fastf1\req.py", line 90, in limit
    raise RateLimitExceededError(self._info)
fastf1.exce

 Season 2018 collecting initiated!


Request for URL https://raw.githubusercontent.com/theOehrly/f1schedule/master/schedule_2018.json failed; using cached response
Traceback (most recent call last):
  File "c:\Users\vitin\VS_CODE\Canal YT\f1-podium-predictor\.venv\Lib\site-packages\requests_cache\session.py", line 311, in _resend
    response = self._send_and_cache(request, actions, cached_response, **kwargs)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\vitin\VS_CODE\Canal YT\f1-podium-predictor\.venv\Lib\site-packages\requests_cache\session.py", line 279, in _send_and_cache
    response = super().send(request, **kwargs)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\vitin\VS_CODE\Canal YT\f1-podium-predictor\.venv\Lib\site-packages\fastf1\req.py", line 129, in send
    lim.limit()
  File "c:\Users\vitin\VS_CODE\Canal YT\f1-podium-predictor\.venv\Lib\site-packages\fastf1\req.py", line 90, in limit
    raise RateLimitExceededError(self._info)
fastf1.exce

 Requisition limit exceeded
Saving collected date until now

 Finished!


Feature Engineering

In [ ]:

import pandas as pd
import numpy as np

# Carregar resultados históricos
df = pd.read_parquet("data/raw/race_results.parquet")

df = df.sort_values(
    ["Date", "Season", "Round"]
).reset_index(drop=True)

# Identificador único de cada GP
df["RaceID"] = (
    df["Season"].astype(str)
    + "_"
    + df["Round"].astype(str)
)

# Variável alvo: conquistou pódio?
df["Podium"] = (
    df["Position"].notna()
    & df["Position"].between(1, 3)
).astype(int)

# 1. Histórico individual do piloto
driver_groups = df.groupby("Driver", sort=False)

df["DriverPodiumRateLast5"] = driver_groups[
    "Podium"
].transform(
    lambda x: x.shift(1).rolling(5, min_periods=1).mean()
)

df["DriverAvgPositionLast5"] = driver_groups[
    "Position"
].transform(
    lambda x: x.shift(1).rolling(5, min_periods=1).mean()
)

# 2. Histórico da equipe
# Primeiro: uma linha por equipe em cada GP
team_races = (
    df.groupby(
        ["Season", "Round", "RaceID", "Date", "Team"],
        as_index=False
    )
    .agg(TeamPodium=("Podium", "max"))
    .sort_values(["Date", "Season", "Round"])
)

team_races["TeamPodiumRateLast5"] = (
    team_races.groupby("Team", sort=False)["TeamPodium"]
    .transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).mean()
    )
)

# Incorporar a feature no dataset de pilotos
df = df.merge(
    team_races[
        ["RaceID", "Team", "TeamPodiumRateLast5"]
    ],
    on=["RaceID", "Team"],
    how="left",
    validate="many_to_one"
)

# 3. Histórico do piloto no circuito
df["CircuitPodiumHistory"] = (
    df.groupby(
        ["Driver", "Circuit"],
        sort=False
    )["Podium"]
    .transform(
        lambda x: x.shift(1).expanding(min_periods=1).mean()
    )
)

# Preencher histórico inexistente
# O piloto não possui histórico anterior naquele contexto
df["DriverPodiumRateLast5"] = (
    df["DriverPodiumRateLast5"].fillna(0)
)

df["TeamPodiumRateLast5"] = (
    df["TeamPodiumRateLast5"].fillna(0)
)

df["CircuitPodiumHistory"] = (
    df["CircuitPodiumHistory"].fillna(0)
)

# Salvar dataset final
df.to_parquet(
    "data/processed/f1_features.parquet",
    index=False
)

print("Feature Engineering concluído!")
print(df.head())

print("\nFeatures disponíveis:")
print(df.columns.tolist())


Temporal Separation

In [ ]:

import pandas as pd
import numpy as np

# Carregar resultados históricos
df = pd.read_parquet("data/raw/race_results.parquet")

df = df.sort_values(
    ["Date", "Season", "Round"]
).reset_index(drop=True)

# Identificador único de cada GP
df["RaceID"] = (
    df["Season"].astype(str)
    + "_"
    + df["Round"].astype(str)
)

# Variável alvo: conquistou pódio?
df["Podium"] = (
    df["Position"].notna()
    & df["Position"].between(1, 3)
).astype(int)

# 1. Histórico individual do piloto
driver_groups = df.groupby("Driver", sort=False)

df["DriverPodiumRateLast5"] = driver_groups[
    "Podium"
].transform(
    lambda x: x.shift(1).rolling(5, min_periods=1).mean()
)

df["DriverAvgPositionLast5"] = driver_groups[
    "Position"
].transform(
    lambda x: x.shift(1).rolling(5, min_periods=1).mean()
)

# 2. Histórico da equipe
# Primeiro: uma linha por equipe em cada GP
team_races = (
    df.groupby(
        ["Season", "Round", "RaceID", "Date", "Team"],
        as_index=False
    )
    .agg(TeamPodium=("Podium", "max"))
    .sort_values(["Date", "Season", "Round"])
)

team_races["TeamPodiumRateLast5"] = (
    team_races.groupby("Team", sort=False)["TeamPodium"]
    .transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).mean()
    )
)

# Incorporar a feature no dataset de pilotos
df = df.merge(
    team_races[
        ["RaceID", "Team", "TeamPodiumRateLast5"]
    ],
    on=["RaceID", "Team"],
    how="left",
    validate="many_to_one"
)

# 3. Histórico do piloto no circuito
df["CircuitPodiumHistory"] = (
    df.groupby(
        ["Driver", "Circuit"],
        sort=False
    )["Podium"]
    .transform(
        lambda x: x.shift(1).expanding(min_periods=1).mean()
    )
)

# Preencher histórico inexistente
# O piloto não possui histórico anterior naquele contexto
df["DriverPodiumRateLast5"] = (
    df["DriverPodiumRateLast5"].fillna(0)
)

df["TeamPodiumRateLast5"] = (
    df["TeamPodiumRateLast5"].fillna(0)
)

df["CircuitPodiumHistory"] = (
    df["CircuitPodiumHistory"].fillna(0)
)

# Salvar dataset final
df.to_parquet(
    "data/processed/f1_features.parquet",
    index=False
)

print("Feature Engineering concluído!")
print(df.head())

print("\nFeatures disponíveis:")
print(df.columns.tolist())


In [ ]:
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report
import pandas as pd

df = pd.read_parquet(
    "data/processed/f1_features.parquet"
)

features = [
    "GridPosition",
    "Driver",
    "Team",
    "Circuit",
    "DriverPodiumRateLast5",
    "DriverAvgPositionLast5",
    "TeamPodiumRateLast5",
    "CircuitPodiumHistory"
]

target = "Podium"

# Separação temporal: treino até 2023
train = df[df["Season"] <= 2023].copy()

# Teste: 2024 e 2025
test = df[df["Season"].isin([2024, 2025])].copy()

X_train = train[features]
y_train = train[target]

X_test = test[features]
y_test = test[target]

categorical_features = [
    "Driver",
    "Team",
    "Circuit"
]

model = CatBoostClassifier(
    iterations=1000,
    depth=6,
    learning_rate=0.03,
    loss_function="Logloss",
    eval_metric="AUC",
    verbose=100,
    random_seed=42
)

model.fit(
    X_train,
    y_train,
    cat_features=categorical_features,
    eval_set=(X_test, y_test),
    early_stopping_rounds=100
)

predictions = model.predict_proba(X_test)[:, 1]

print(classification_report(
    y_test,
    (predictions >= 0.5).astype(int)
))